# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,896,000 new records


write_files: 2,896,000 records -> /home/jovyan/data/lake/logs/run-20261119-123155-aacfdb



Lake check — topic 'logs' (4 partitions)   2026-11-19 12:32:49
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 716,854)     716,854           0          0           0
        1           [0, 730,228)     730,228           0          0           0
        2           [0, 718,944)     718,944           0          0           0
        3           [0, 729,974)     729,974           0          0           0
-------------------------------------------------------------------------------
    total      2,896,000 records   2,896,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 161.3 MB, 40.32 MB per file on average
check took 0.4 s

Exactly once: PASS


{'topic': 2896000,
 'lake': 2896000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 161.298651,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,896,000 new records


write_files: 2,896,000 records -> /home/jovyan/data/lake/logs/run-20261119-123250-04483f



Lake check — topic 'logs' (4 partitions)   2026-11-19 12:33:33
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 716,854)   1,433,708     716,854          0           0
        1           [0, 730,228)   1,460,456     730,228          0           0
        2           [0, 718,944)   1,437,888     718,944          0           0
        3           [0, 729,974)   1,459,948     729,974          0           0
-------------------------------------------------------------------------------
    total      2,896,000 records   5,792,000   2,896,000          0           0

files under /home/jovyan/data/lake: 8 data files, 322.6 MB, 40.32 MB per file on average
check took 0.8 s

Exactly once: FAIL — 2,896,000 duplicates


{'topic': 2896000,
 'lake': 5792000,
 'dup': 2896000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 322.597302,
 'ok': False,
 'problems': ['2,896,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [4]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it to staging
    run = write_files(new, STAGING)
    # 4. move the finished run into the lake in one step
    publish(run)

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [5]:
check_all(run_once)

reset: lake, staging and state deleted



situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,896,000           0        0           0  38.1s  PASS
2 new records arrive, run again         2,996,000           0        0           0   7.3s  PASS
3 nothing new, run again                2,996,000           0        0           0   0.5s  PASS
4 run stops WHILE writing: right after  2,996,000           0        0     100,000         PASS
  ... then the next run                 3,096,000           0        0           0   9.1s  PASS
5 run stops AFTER writing: right after  3,096,000           0        0     100,000         PASS
  ... then the next run                 3,196,000           0        0           0   8.4s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: YES


[('1 first run',
  {'topic': 2896000,
   'lake': 2896000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 161.298651,
   'ok': True,
   'problems': []},
  38.119921922683716),
 ('2 new records arrive, run again',
  {'topic': 2996000,
   'lake': 2996000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 162.422436,
   'ok': True,
   'problems': []},
  7.331404447555542),
 ('3 nothing new, run again',
  {'topic': 2996000,
   'lake': 2996000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 162.422436,
   'ok': True,
   'problems': []},
  0.45293712615966797),
 ('4 run stops WHILE writing: right after',
  {'topic': 3096000,
   'lake': 2996000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 162.422436,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 3096000,
   'lake': 3096000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 163.527954,
   'ok': True,
   'problems': []},